# 🔧 MÓDULO 2: Preparación de Datos
## Feature Engineering para Machine Learning

**Objetivo:** Transformar variables categóricas y preparar el dataset para entrenar modelos.

---

## 2.1 Configuración

In [1]:
import sys

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("📍 Ejecutando en Google Colab")
    RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/'
else:
    print("🐳 Ejecutando en Docker")
    RUTA_DATOS = '../data/'

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Vuelos_Modulo2") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} iniciado")

🐳 Ejecutando en Docker
✅ Spark 3.5.0 iniciado


In [2]:
# Cargar datos
df = spark.read.csv(RUTA_DATOS + "flights_2015_full.csv", header=True, inferSchema=True)
print(f"✈️ {df.count():,} vuelos cargados")

✈️ 5,332,914 vuelos cargados


## 2.2 Entendiendo el Problema

**Variables categóricas** (texto) que debemos convertir a números:
- `AIRLINE` → 14 aerolíneas diferentes
- `ORIGIN` → ~300 aeropuertos
- `DEST` → ~300 aeropuertos

**Variables numéricas** (ya listas):
- `MONTH`, `DAY_OF_WEEK`, `DEP_HOUR`, `DISTANCE`

**Variable objetivo:**
- `DEP_DEL15` → 0 (puntual) o 1 (retrasado ≥15 min)

In [3]:
# Ver cuántas categorías tiene cada variable
print("📊 CARDINALIDAD DE VARIABLES CATEGÓRICAS")
print(f"  AIRLINE: {df.select('AIRLINE').distinct().count()} categorías")
print(f"  ORIGIN:  {df.select('ORIGIN').distinct().count()} categorías")
print(f"  DEST:    {df.select('DEST').distinct().count()} categorías")

📊 CARDINALIDAD DE VARIABLES CATEGÓRICAS
  AIRLINE: 14 categorías
  ORIGIN:  322 categorías
  DEST:    322 categorías


## 2.3 StringIndexer: Convertir Texto a Números

`StringIndexer` asigna un número a cada categoría basado en frecuencia:
- La categoría más frecuente → 0
- La segunda más frecuente → 1
- etc.

In [4]:
from pyspark.ml.feature import StringIndexer

# Crear indexers para cada variable categórica
indexer_airline = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_idx", handleInvalid="keep")
indexer_origin = StringIndexer(inputCol="ORIGIN", outputCol="ORIGIN_idx", handleInvalid="keep")
indexer_dest = StringIndexer(inputCol="DEST", outputCol="DEST_idx", handleInvalid="keep")

print("✅ Indexers creados")

✅ Indexers creados


In [5]:
# Aplicar transformaciones
df_idx = indexer_airline.fit(df).transform(df)
df_idx = indexer_origin.fit(df_idx).transform(df_idx)
df_idx = indexer_dest.fit(df_idx).transform(df_idx)

print("✅ Variables indexadas")
df_idx.select("AIRLINE", "AIRLINE_idx", "ORIGIN", "ORIGIN_idx").show(5)

✅ Variables indexadas
+-------+-----------+------+----------+
|AIRLINE|AIRLINE_idx|ORIGIN|ORIGIN_idx|
+-------+-----------+------+----------+
|     AS|        9.0|   ANC|      61.0|
|     AA|        2.0|   LAX|       4.0|
|     US|        8.0|   SFO|       5.0|
|     AA|        2.0|   LAX|       4.0|
|     AS|        9.0|   SEA|      11.0|
+-------+-----------+------+----------+
only showing top 5 rows



## 2.4 VectorAssembler: Crear Vector de Features

Los modelos de Spark MLlib necesitan un **vector** con todas las características.

In [6]:
from pyspark.ml.feature import VectorAssembler

# Definir qué columnas usar como features
feature_cols = [
    "MONTH",
    "DAY_OF_WEEK",
    "DEP_HOUR",
    "DISTANCE",
    "AIRLINE_idx",
    "ORIGIN_idx",
    "DEST_idx"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_final = assembler.transform(df_idx)
print("✅ Vector de features creado")

✅ Vector de features creado


In [7]:
# Ver resultado
df_final.select("features", "DEP_DEL15").show(5, truncate=False)

+----------------------------------+---------+
|features                          |DEP_DEL15|
+----------------------------------+---------+
|[1.0,4.0,0.0,1448.0,9.0,61.0,11.0]|0        |
|[1.0,4.0,0.0,2330.0,2.0,4.0,50.0] |0        |
|[1.0,4.0,0.0,2296.0,8.0,5.0,15.0] |0        |
|[1.0,4.0,0.0,2342.0,2.0,4.0,24.0] |0        |
|[1.0,4.0,0.0,1448.0,9.0,11.0,61.0]|0        |
+----------------------------------+---------+
only showing top 5 rows



## 2.5 División Train/Test

Dividimos 80% para entrenar, 20% para evaluar.

In [8]:
# Seleccionar solo columnas necesarias
df_ml = df_final.select("features", "DEP_DEL15")

# Dividir
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"📊 DIVISIÓN DE DATOS")
print(f"  Entrenamiento: {train_df.count():,} vuelos ({train_df.count()/df_ml.count()*100:.1f}%)")
print(f"  Prueba:        {test_df.count():,} vuelos ({test_df.count()/df_ml.count()*100:.1f}%)")

📊 DIVISIÓN DE DATOS
  Entrenamiento: 4,266,480 vuelos (80.0%)
  Prueba:        1,066,434 vuelos (20.0%)


In [9]:
# Verificar balance de clases
from pyspark.sql.functions import col, count

print("\n📊 BALANCE DE CLASES EN TRAIN:")
train_df.groupBy("DEP_DEL15").count().show()


📊 BALANCE DE CLASES EN TRAIN:
+---------+-------+
|DEP_DEL15|  count|
+---------+-------+
|        1| 797739|
|        0|3468741|
+---------+-------+



## 2.6 Guardar Datos Preparados (Opcional)

Para no repetir la preparación en cada módulo.

In [10]:
# Guardar para uso posterior
train_df.write.parquet(RUTA_DATOS + "train_prepared.parquet", mode="overwrite")
test_df.write.parquet(RUTA_DATOS + "test_prepared.parquet", mode="overwrite")
print("✅ Datos guardados")

print("💡 Descomenta las líneas anteriores si quieres guardar los datos preparados")

✅ Datos guardados
💡 Descomenta las líneas anteriores si quieres guardar los datos preparados


---
## ✅ CHECKPOINT MÓDULO 2

Ahora tienes:

| Variable | Descripción |
|----------|-------------|
| `train_df` | Dataset de entrenamiento con features |
| `test_df` | Dataset de prueba para evaluar |
| `feature_cols` | Lista de nombres de features |

**Siguiente:** → `Modulo3_Modelos.ipynb`